In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

c:\Users\mayan\Downloads\Weather_Forecasting


In [3]:
from backend.cities import CITIES

print("Total cities:", len(CITIES))

Total cities: 35


In [4]:
for city, data in CITIES.items():
    print(city, "→", data["state"])

Amaravati → Andhra Pradesh
Itanagar → Arunachal Pradesh
Dispur → Assam
Patna → Bihar
Raipur → Chhattisgarh
Panaji → Goa
Gandhinagar → Gujarat
Chandigarh → Haryana / Punjab / Union Territory
Shimla → Himachal Pradesh
Ranchi → Jharkhand
Bengaluru → Karnataka
Thiruvananthapuram → Kerala
Bhopal → Madhya Pradesh
Mumbai → Maharashtra
Imphal → Manipur
Shillong → Meghalaya
Aizawl → Mizoram
Kohima → Nagaland
Bhubaneswar → Odisha
Jaipur → Rajasthan
Gangtok → Sikkim
Chennai → Tamil Nadu
Hyderabad → Telangana
Agartala → Tripura
Lucknow → Uttar Pradesh
Dehradun → Uttarakhand
Kolkata → West Bengal
Port Blair → Andaman and Nicobar Islands
Daman → Dadra and Nagar Haveli and Daman and Diu
New Delhi → Delhi
Srinagar → Jammu and Kashmir
Jammu → Jammu and Kashmir
Leh → Ladakh
Kavaratti → Lakshadweep
Puducherry → Puducherry


In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.live_weather import CITIES

print("Total cities:", len(CITIES))
print("Bhopal:", CITIES["Bhopal"])
print("Amaravati:", CITIES["Amaravati"])

Total cities: 35
Bhopal: {'state': 'Madhya Pradesh', 'latitude': 23.2599, 'longitude': 77.4126, 'type': 'State Capital'}
Amaravati: {'state': 'Andhra Pradesh', 'latitude': 16.5742, 'longitude': 80.357, 'type': 'State Capital'}


In [8]:
import pandas as pd

# Load dataset
df = pd.read_csv("../data/raw/Weather_Forecasting.csv")

# Convert time
df["time"] = pd.to_datetime(df["time"], format="mixed")

# Sort
df = df.sort_values(["city", "time"]).reset_index(drop=True)

# Check each city
for city in df["city"].unique():

    city_df = df[df["city"] == city].copy()
    city_df = city_df.sort_values("time")

    time_diff = city_df["time"].diff().dropna()

    expected_gap = pd.Timedelta(hours=1)

    missing_gaps = time_diff[time_diff != expected_gap]

    print("=" * 50)
    print("City:", city)
    print("Rows:", len(city_df))
    print("Start:", city_df["time"].min())
    print("End:", city_df["time"].max())
    print("Non-1-hour gaps:", len(missing_gaps))

    if len(missing_gaps) > 0:
        print(missing_gaps.head())

City: Bengaluru
Rows: 41472
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00
Non-1-hour gaps: 0
City: Bhopal
Rows: 65424
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00
Non-1-hour gaps: 23952
58993   0 days
58995   0 days
58997   0 days
58999   0 days
59001   0 days
Name: time, dtype: timedelta64[ns]
City: Chennai
Rows: 41472
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00
Non-1-hour gaps: 0
City: Delhi
Rows: 41472
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00
Non-1-hour gaps: 0
City: Mumbai
Rows: 41472
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00
Non-1-hour gaps: 0


In [9]:
# Check duplicate timestamps for Bhopal

bhopal = df[df["city"] == "Bhopal"].copy()

duplicates = bhopal[
    bhopal.duplicated(subset=["time"], keep=False)
].sort_values("time")

print("Total Bhopal rows:", len(bhopal))
print("Unique timestamps:", bhopal["time"].nunique())
print("Duplicate rows:", len(duplicates))

print("\nFirst duplicate timestamps:")
print(
    duplicates[["time", "temperature_2m", "relative_humidity_2m"]]
    .head(20)
)

Total Bhopal rows: 65424
Unique timestamps: 41472
Duplicate rows: 47904

First duplicate timestamps:
                     time  temperature_2m  relative_humidity_2m
58992 2024-01-01 00:00:00            16.5                    91
58993 2024-01-01 00:00:00            16.5                    91
58994 2024-01-01 01:00:00            16.0                    93
58995 2024-01-01 01:00:00            16.0                    93
58996 2024-01-01 02:00:00            15.8                    92
58997 2024-01-01 02:00:00            15.8                    92
58998 2024-01-01 03:00:00            15.5                    92
58999 2024-01-01 03:00:00            15.5                    92
59000 2024-01-01 04:00:00            14.9                    93
59001 2024-01-01 04:00:00            14.9                    93
59003 2024-01-01 05:00:00            14.8                    93
59002 2024-01-01 05:00:00            14.8                    93
59004 2024-01-01 06:00:00            14.2                    94
590

In [10]:
# Check how many times each timestamp occurs

duplicate_counts = (
    bhopal.groupby("time")
    .size()
    .reset_index(name="count")
)

print(
    duplicate_counts[
        duplicate_counts["count"] > 1
    ].head(20)
)

                     time  count
17520 2024-01-01 00:00:00      2
17521 2024-01-01 01:00:00      2
17522 2024-01-01 02:00:00      2
17523 2024-01-01 03:00:00      2
17524 2024-01-01 04:00:00      2
17525 2024-01-01 05:00:00      2
17526 2024-01-01 06:00:00      2
17527 2024-01-01 07:00:00      2
17528 2024-01-01 08:00:00      2
17529 2024-01-01 09:00:00      2
17530 2024-01-01 10:00:00      2
17531 2024-01-01 11:00:00      2
17532 2024-01-01 12:00:00      2
17533 2024-01-01 13:00:00      2
17534 2024-01-01 14:00:00      2
17535 2024-01-01 15:00:00      2
17536 2024-01-01 16:00:00      2
17537 2024-01-01 17:00:00      2
17538 2024-01-01 18:00:00      2
17539 2024-01-01 19:00:00      2


In [11]:
bhopal = df[df["city"] == "Bhopal"].copy()

# Check whether duplicate timestamps also have identical data
duplicate_data = bhopal[bhopal.duplicated(keep=False)].sort_values("time")

print("Total duplicate records:", len(duplicate_data))
print("Exact duplicate rows:",
      bhopal.duplicated(keep=False).sum())

print("\nNumber of unique rows:")
print(bhopal.drop_duplicates().shape[0])

Total duplicate records: 47904
Exact duplicate rows: 47904

Number of unique rows:
41472


In [12]:
# Check how many records exist for each timestamp
duplicate_counts = (
    bhopal.groupby("time")
    .size()
    .reset_index(name="count")
)

print(duplicate_counts["count"].value_counts().sort_index())

count
1    17520
2    23952
Name: count, dtype: int64


In [13]:
# Remove exact duplicate rows
df_clean = df.drop_duplicates().copy()

print("Original rows:", len(df))
print("Cleaned rows:", len(df_clean))
print("Rows removed:", len(df) - len(df_clean))

Original rows: 231312
Cleaned rows: 207360
Rows removed: 23952


In [14]:
for city in df_clean["city"].unique():
    city_df = (
        df_clean[df_clean["city"] == city]
        .sort_values("time")
    )

    time_diff = city_df["time"].diff().dropna()
    gaps = time_diff[time_diff != pd.Timedelta(hours=1)]

    print(
        city,
        "| rows:", len(city_df),
        "| start:", city_df["time"].min(),
        "| end:", city_df["time"].max(),
        "| non-1-hour gaps:", len(gaps)
    )

Bengaluru | rows: 41472 | start: 2022-01-01 00:00:00 | end: 2026-09-24 23:00:00 | non-1-hour gaps: 0
Bhopal | rows: 41472 | start: 2022-01-01 00:00:00 | end: 2026-09-24 23:00:00 | non-1-hour gaps: 0
Chennai | rows: 41472 | start: 2022-01-01 00:00:00 | end: 2026-09-24 23:00:00 | non-1-hour gaps: 0
Delhi | rows: 41472 | start: 2022-01-01 00:00:00 | end: 2026-09-24 23:00:00 | non-1-hour gaps: 0
Mumbai | rows: 41472 | start: 2022-01-01 00:00:00 | end: 2026-09-24 23:00:00 | non-1-hour gaps: 0


In [15]:
# Save the cleaned dataset
df_clean.to_csv("Weather_Forecasting_clean.csv", index=False)

print("Cleaned dataset saved successfully.")
print("Rows:", len(df_clean))
print("Columns:", len(df_clean.columns))

Cleaned dataset saved successfully.
Rows: 207360
Columns: 16


In [16]:
import pandas as pd
import requests
import time
from pathlib import Path

from backend.cities import CITIES

# ============================================================
# SETTINGS
# ============================================================

START_DATE = "2022-01-01"
END_DATE = "2026-09-24"

# Exact weather columns used by your existing dataset
HOURLY_VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
    "wind_gusts_10m",
    "wind_direction_10m"
]

# Existing cities already present in the cleaned dataset
EXISTING_CITIES = {
    "Bengaluru",
    "Bhopal",
    "Chennai",
    "Delhi",
    "Mumbai"
}

# ============================================================
# FIND MISSING CITIES
# ============================================================

missing_cities = []

for city, info in CITIES.items():

    # "New Delhi" in cities.py corresponds to "Delhi" in CSV
    if city == "New Delhi":
        if "Delhi" not in EXISTING_CITIES:
            missing_cities.append(city)

    elif city not in EXISTING_CITIES:
        missing_cities.append(city)

print("Total cities in registry:", len(CITIES))
print("Existing cities:", len(EXISTING_CITIES))
print("Missing cities to download:", len(missing_cities))

print("\nMissing cities:")
for city in missing_cities:
    print("-", city)

Total cities in registry: 35
Existing cities: 5
Missing cities to download: 30

Missing cities:
- Amaravati
- Itanagar
- Dispur
- Patna
- Raipur
- Panaji
- Gandhinagar
- Chandigarh
- Shimla
- Ranchi
- Thiruvananthapuram
- Imphal
- Shillong
- Aizawl
- Kohima
- Bhubaneswar
- Jaipur
- Gangtok
- Hyderabad
- Agartala
- Lucknow
- Dehradun
- Kolkata
- Port Blair
- Daman
- Srinagar
- Jammu
- Leh
- Kavaratti
- Puducherry


In [17]:
all_new_city_data = []

for i, city in enumerate(missing_cities, start=1):

    info = CITIES[city]

    # Use Delhi as the dataset name for New Delhi
    dataset_city_name = "Delhi" if city == "New Delhi" else city

    print(f"\n[{i}/{len(missing_cities)}] Downloading {city}...")

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": info["latitude"],
        "longitude": info["longitude"],
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": ",".join(HOURLY_VARIABLES),
        "timezone": "Asia/Kolkata",
        "temperature_unit": "celsius",
        "wind_speed_unit": "kmh",
        "precipitation_unit": "mm"
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=120
        )

        response.raise_for_status()

        data = response.json()

        if "hourly" not in data:
            print(f"ERROR: No hourly data returned for {city}")
            continue

        hourly = data["hourly"]

        city_df = pd.DataFrame(hourly)

        # Add city information
        city_df["city"] = dataset_city_name
        city_df["state"] = info["state"]
        city_df["latitude"] = info["latitude"]
        city_df["longitude"] = info["longitude"]

        # Convert time
        city_df["time"] = pd.to_datetime(city_df["time"])

        # Make sure columns have the same order as the existing dataset
        city_df = city_df[
            [
                "time",
                "temperature_2m",
                "relative_humidity_2m",
                "dew_point_2m",
                "apparent_temperature",
                "precipitation",
                "rain",
                "surface_pressure",
                "cloud_cover",
                "wind_speed_10m",
                "wind_gusts_10m",
                "wind_direction_10m",
                "city",
                "state",
                "latitude",
                "longitude"
            ]
        ]

        all_new_city_data.append(city_df)

        print("Downloaded rows:", len(city_df))
        print("Start:", city_df["time"].min())
        print("End:", city_df["time"].max())

    except Exception as e:
        print(f"ERROR downloading {city}: {e}")

    # Small pause between requests
    time.sleep(1)


[1/30] Downloading Amaravati...
Downloaded rows: 41472
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00

[2/30] Downloading Itanagar...
Downloaded rows: 41472
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00

[3/30] Downloading Dispur...
Downloaded rows: 41472
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00

[4/30] Downloading Patna...
Downloaded rows: 41472
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00

[5/30] Downloading Raipur...
Downloaded rows: 41472
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00

[6/30] Downloading Panaji...
Downloaded rows: 41472
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00

[7/30] Downloading Gandhinagar...
Downloaded rows: 41472
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00

[8/30] Downloading Chandigarh...
Downloaded rows: 41472
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00

[9/30] Downloading Shimla...
Downloaded rows: 41472
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00

[10/30] Downloading Ranchi...
Downloaded 

In [18]:
print("Successfully downloaded cities:", len(all_new_city_data))

for df_city in all_new_city_data:
    print(
        df_city["city"].iloc[0],
        "->",
        len(df_city),
        "rows"
    )

Successfully downloaded cities: 15
Amaravati -> 41472 rows
Itanagar -> 41472 rows
Dispur -> 41472 rows
Patna -> 41472 rows
Raipur -> 41472 rows
Panaji -> 41472 rows
Gandhinagar -> 41472 rows
Chandigarh -> 41472 rows
Shimla -> 41472 rows
Ranchi -> 41472 rows
Jaipur -> 41472 rows
Gangtok -> 41472 rows
Hyderabad -> 41472 rows
Agartala -> 41472 rows
Lucknow -> 41472 rows


In [19]:
downloaded_cities = {
    df_city["city"].iloc[0]
    for df_city in all_new_city_data
}

remaining_cities = [
    city for city in missing_cities
    if ("Delhi" if city == "New Delhi" else city) not in downloaded_cities
]

print("Already downloaded:", len(downloaded_cities))
print("Remaining:", len(remaining_cities))

print("\nRemaining cities:")
for city in remaining_cities:
    print("-", city)

Already downloaded: 15
Remaining: 15

Remaining cities:
- Thiruvananthapuram
- Imphal
- Shillong
- Aizawl
- Kohima
- Bhubaneswar
- Dehradun
- Kolkata
- Port Blair
- Daman
- Srinagar
- Jammu
- Leh
- Kavaratti
- Puducherry


In [20]:
import pandas as pd
import requests
import time

new_downloads = []

for i, city in enumerate(remaining_cities, start=1):

    info = CITIES[city]

    print(f"\n[{i}/{len(remaining_cities)}] Downloading {city}...")

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": info["latitude"],
        "longitude": info["longitude"],
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": ",".join(HOURLY_VARIABLES),
        "timezone": "Asia/Kolkata",
        "temperature_unit": "celsius",
        "wind_speed_unit": "kmh",
        "precipitation_unit": "mm"
    }

    success = False

    for attempt in range(3):

        try:
            response = requests.get(
                url,
                params=params,
                timeout=120
            )

            if response.status_code == 429:
                print("Rate limit reached.")
                print("Waiting 30 seconds before retry...")
                time.sleep(30)
                continue

            response.raise_for_status()

            data = response.json()

            if "hourly" not in data:
                print(f"ERROR: No hourly data returned for {city}")
                break

            hourly = data["hourly"]

            city_df = pd.DataFrame(hourly)

            # Keep Delhi naming consistent if needed
            dataset_city_name = "Delhi" if city == "New Delhi" else city

            city_df["city"] = dataset_city_name
            city_df["state"] = info["state"]
            city_df["latitude"] = info["latitude"]
            city_df["longitude"] = info["longitude"]

            city_df["time"] = pd.to_datetime(city_df["time"])

            city_df = city_df[
                [
                    "time",
                    "temperature_2m",
                    "relative_humidity_2m",
                    "dew_point_2m",
                    "apparent_temperature",
                    "precipitation",
                    "rain",
                    "surface_pressure",
                    "cloud_cover",
                    "wind_speed_10m",
                    "wind_gusts_10m",
                    "wind_direction_10m",
                    "city",
                    "state",
                    "latitude",
                    "longitude"
                ]
            ]

            new_downloads.append(city_df)

            print("Downloaded rows:", len(city_df))
            print("Start:", city_df["time"].min())
            print("End:", city_df["time"].max())

            success = True
            break

        except Exception as e:
            print(f"Attempt {attempt + 1} failed:", e)

            if attempt < 2:
                print("Waiting 30 seconds before retry...")
                time.sleep(30)

    if not success:
        print(f"FAILED: {city}")

    # Pause before the next city
    print("Waiting 20 seconds before next city...")
    time.sleep(20)


[1/15] Downloading Thiruvananthapuram...
Downloaded rows: 41472
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00
Waiting 20 seconds before next city...

[2/15] Downloading Imphal...
Downloaded rows: 41472
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00
Waiting 20 seconds before next city...

[3/15] Downloading Shillong...
Downloaded rows: 41472
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00
Waiting 20 seconds before next city...

[4/15] Downloading Aizawl...
Downloaded rows: 41472
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00
Waiting 20 seconds before next city...

[5/15] Downloading Kohima...
Downloaded rows: 41472
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00
Waiting 20 seconds before next city...

[6/15] Downloading Bhubaneswar...
Downloaded rows: 41472
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00
Waiting 20 seconds before next city...

[7/15] Downloading Dehradun...
Downloaded rows: 41472
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00
Waiting 20

In [21]:
print("New cities downloaded:", len(new_downloads))

for df_city in new_downloads:
    print(df_city["city"].iloc[0], "->", len(df_city), "rows")

New cities downloaded: 15
Thiruvananthapuram -> 41472 rows
Imphal -> 41472 rows
Shillong -> 41472 rows
Aizawl -> 41472 rows
Kohima -> 41472 rows
Bhubaneswar -> 41472 rows
Dehradun -> 41472 rows
Kolkata -> 41472 rows
Port Blair -> 41472 rows
Daman -> 41472 rows
Srinagar -> 41472 rows
Jammu -> 41472 rows
Leh -> 41472 rows
Kavaratti -> 41472 rows
Puducherry -> 41472 rows


In [22]:
# Combine the original 5 cleaned cities with the 30 newly downloaded cities

df_final = pd.concat(
    [
        df_clean,
        pd.concat(all_new_city_data, ignore_index=True),
        pd.concat(new_downloads, ignore_index=True)
    ],
    ignore_index=True
)

print("Final rows:", len(df_final))
print("Final columns:", len(df_final.columns))

print("\nCities:")
print(df_final["city"].nunique())

print("\nRows per city:")
print(df_final["city"].value_counts().sort_index())

Final rows: 1451520
Final columns: 16

Cities:
35

Rows per city:
city
Agartala              41472
Aizawl                41472
Amaravati             41472
Bengaluru             41472
Bhopal                41472
Bhubaneswar           41472
Chandigarh            41472
Chennai               41472
Daman                 41472
Dehradun              41472
Delhi                 41472
Dispur                41472
Gandhinagar           41472
Gangtok               41472
Hyderabad             41472
Imphal                41472
Itanagar              41472
Jaipur                41472
Jammu                 41472
Kavaratti             41472
Kohima                41472
Kolkata               41472
Leh                   41472
Lucknow               41472
Mumbai                41472
Panaji                41472
Patna                 41472
Port Blair            41472
Puducherry            41472
Raipur                41472
Ranchi                41472
Shillong              41472
Shimla                41472
Srina

In [23]:
print("Checking duplicate rows...")

duplicate_rows = df_final.duplicated().sum()

print("Duplicate rows:", duplicate_rows)

Checking duplicate rows...
Duplicate rows: 0


In [24]:
duplicate_times = df_final.duplicated(
    subset=["city", "time"]
).sum()

print("Duplicate city-time records:", duplicate_times)

Duplicate city-time records: 0


In [25]:
id="x8k2p1"
print("Checking hourly continuity for all cities...\n")

gap_results = []

for city in sorted(df_final["city"].unique()):

    city_df = df_final[df_final["city"] == city].copy()
    city_df = city_df.sort_values("time")

    time_diff = city_df["time"].diff()

    gaps = (time_diff != pd.Timedelta(hours=1)).sum()

    # The first row naturally has no previous timestamp
    gaps = gaps - 1

    gap_results.append({
        "city": city,
        "rows": len(city_df),
        "gaps": gaps,
        "start": city_df["time"].min(),
        "end": city_df["time"].max()
    })

gap_check = pd.DataFrame(gap_results)

print(gap_check.to_string(index=False))

Checking hourly continuity for all cities...

              city  rows  gaps      start                 end
          Agartala 41472     0 2022-01-01 2026-09-24 23:00:00
            Aizawl 41472     0 2022-01-01 2026-09-24 23:00:00
         Amaravati 41472     0 2022-01-01 2026-09-24 23:00:00
         Bengaluru 41472     0 2022-01-01 2026-09-24 23:00:00
            Bhopal 41472     0 2022-01-01 2026-09-24 23:00:00
       Bhubaneswar 41472     0 2022-01-01 2026-09-24 23:00:00
        Chandigarh 41472     0 2022-01-01 2026-09-24 23:00:00
           Chennai 41472     0 2022-01-01 2026-09-24 23:00:00
             Daman 41472     0 2022-01-01 2026-09-24 23:00:00
          Dehradun 41472     0 2022-01-01 2026-09-24 23:00:00
             Delhi 41472     0 2022-01-01 2026-09-24 23:00:00
            Dispur 41472     0 2022-01-01 2026-09-24 23:00:00
       Gandhinagar 41472     0 2022-01-01 2026-09-24 23:00:00
           Gangtok 41472     0 2022-01-01 2026-09-24 23:00:00
         Hyderabad 41472

In [26]:
print("Checking missing values...\n")

missing_values = df_final.isnull().sum()

print(missing_values)

Checking missing values...

time                    0
temperature_2m          0
relative_humidity_2m    0
dew_point_2m            0
apparent_temperature    0
precipitation           0
rain                    0
surface_pressure        0
cloud_cover             0
wind_speed_10m          0
wind_direction_10m      0
wind_gusts_10m          0
city                    0
state                   0
latitude                0
longitude               0
dtype: int64


In [27]:
total_missing = df_final.isnull().sum().sum()

print("\nTotal missing values:", total_missing)


Total missing values: 0


In [28]:
print("========== FINAL DATASET VALIDATION ==========")

print("Rows:", len(df_final))
print("Columns:", len(df_final.columns))
print("Cities:", df_final["city"].nunique())

print("\nDate range:")
print("Start:", df_final["time"].min())
print("End:", df_final["time"].max())

print("\nColumns:")
for col in df_final.columns:
    print("-", col)

print("\nMissing values:", df_final.isnull().sum().sum())
print("Duplicate rows:", df_final.duplicated().sum())
print(
    "Duplicate city-time:",
    df_final.duplicated(subset=["city", "time"]).sum()
)

print("\n==============================================")

========== FINAL DATASET VALIDATION ==========
Rows: 1451520
Columns: 16
Cities: 35

Date range:
Start: 2022-01-01 00:00:00
End: 2026-09-24 23:00:00

Columns:
- time
- temperature_2m
- relative_humidity_2m
- dew_point_2m
- apparent_temperature
- precipitation
- rain
- surface_pressure
- cloud_cover
- wind_speed_10m
- wind_direction_10m
- wind_gusts_10m
- city
- state
- latitude
- longitude

Missing values: 0
Duplicate rows: 0
Duplicate city-time: 0



In [29]:
FINAL_FILE = "Weather_Forecasting_final.csv"

df_final.to_csv(FINAL_FILE, index=False)

print("Final dataset saved successfully!")
print("File:", FINAL_FILE)
print("Rows:", len(df_final))
print("Columns:", len(df_final.columns))

Final dataset saved successfully!
File: Weather_Forecasting_final.csv
Rows: 1451520
Columns: 16
